In [ ]:

!pip install diffusers transformers open_clip_torch accelerate -q

import torch, os, glob, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import open_clip
from torchvision import transforms
from PIL import Image
from diffusers import AutoencoderKL
warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device: " + device.upper())
if device == "cuda":
    print("GPU: " + torch.cuda.get_device_name(0))
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print("VRAM: " + str(round(mem,1)) + " GB")


In [ ]:

vae = AutoencoderKL.from_pretrained(
    "stabilityai/sd-vae-ft-mse",
    torch_dtype=torch.float32
).to(device).eval()
for p in vae.parameters():
    p.requires_grad = False
print("VAE loaded and frozen")

# Latent space dimensions
# VAE compresses 512x512 image -> 64x64x4 latent (8x smaller)
LATENT_H = 64
LATENT_W = 64
LATENT_C = 4
print("Latent shape: " + str(LATENT_C) + "x" + str(LATENT_H) + "x" + str(LATENT_W))

# Load frozen CLIP
print("Loading CLIP...")
clip_model, _, _ = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
clip_model = clip_model.to(device).eval()
tokenizer = open_clip.get_tokenizer("ViT-B-32")
for p in clip_model.parameters():
    p.requires_grad = False

CLIP_MEAN = torch.tensor([0.48145466,0.4578275,0.40821073]).view(1,3,1,1).to(device)
CLIP_STD  = torch.tensor([0.26862954,0.26130258,0.27577711]).view(1,3,1,1).to(device)

total_params = sum(p.numel() for p in list(vae.parameters()) + list(clip_model.parameters()))
print("Total frozen params: " + str(total_params))


In [ ]:


IMG_SIZE = 512
DATASET  = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/train/NORMAL"

if not os.path.exists(DATASET):
    raise FileNotFoundError("Dataset not found! Check path.")

paths = glob.glob(DATASET + "/*.jpeg") + glob.glob(DATASET + "/*.jpg")
random.seed(42)
paths = random.sample(paths, min(100, len(paths)))

tfm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor()
])

imgs = []
for p in paths:
    try:
        imgs.append(tfm(Image.open(p).convert("RGB")))
    except:
        pass
X_med = torch.stack(imgs).to(device)
print("Loaded " + str(len(X_med)) + " real X-rays")

# Extract CLIP features for kernel
def get_clip_feats(imgs_b):
    n = (imgs_b.clamp(0,1) - CLIP_MEAN) / CLIP_STD
    f = clip_model.encode_image(n)
    return f / f.norm(dim=-1, keepdim=True)

print("Building kernel...")
with torch.no_grad():
    feats = torch.cat([get_clip_feats(X_med[i:i+20]) for i in range(0,len(X_med),20)])
    diff  = feats.unsqueeze(0) - feats.unsqueeze(1)
    K     = torch.exp(-(diff**2).sum(-1) / (2*0.5**2))
    alpha = torch.linalg.solve(
        K + 1e-3*torch.eye(len(X_med), device=device),
        torch.ones(len(X_med), 1, device=device)
    )


In [ ]:


SIGMA = 0.5

def decode_latent(z):
    with torch.no_grad():
        img = vae.decode(z / 0.18215).sample
        img = (img / 2 + 0.5).clamp(0, 1)
    return img

def latent_score_fn(z, prompt):
    z_in = z.detach().clone().requires_grad_(True)

    img = vae.decode(z_in / 0.18215).sample
    img = (img / 2 + 0.5).clamp(0, 1)

    img_224 = torch.nn.functional.interpolate(img, (224,224), mode="bilinear", align_corners=False)
    img_norm = (img_224 - CLIP_MEAN) / CLIP_STD

    tok = tokenizer([prompt]).to(device)
    i_f = clip_model.encode_image(img_norm)
    t_f = clip_model.encode_text(tok)
    i_f = i_f / i_f.norm(dim=-1, keepdim=True)
    t_f = t_f / t_f.norm(dim=-1, keepdim=True)
    clip_s = (i_f * t_f).sum() * 3.0      # lowered from 8.0

    phi   = clip_model.encode_image(img_norm)
    phi   = phi / phi.norm(dim=-1, keepdim=True)
    k_v   = torch.exp(-(((phi - feats)**2).sum(-1)) / (2*SIGMA**2))
    kern_s = (alpha.squeeze() * k_v).sum() * 2.5   # raised from 1.5

    (clip_s + kern_s).backward()
    return z_in.grad.detach()


In [ ]:


PROMPTS = [
    "chest X-ray showing healthy lungs, grayscale, medical imaging",
    "chest X-ray showing pneumonia infection in lungs, grayscale, medical imaging",
    "chest X-ray showing lung nodule, grayscale, medical imaging"
]
SEEDS  = [42, 7, 123]
STEPS  = 500           # more steps, slower refinement
LR     = 0.04          # smaller steps = more stable
NOISE  = 0.03          # less noise = cleaner result

results = []

for idx, (prompt, seed) in enumerate(zip(PROMPTS, SEEDS)):
    print("Generating image " + str(idx+1) + "/3")
    print("Prompt: " + prompt[:60])

    torch.manual_seed(seed)
    z = torch.randn(1, LATENT_C, LATENT_H, LATENT_W).to(device) * 0.5
    sims = []
    tok  = tokenizer([prompt]).to(device)

    for t in range(STEPS + 1):
        noise_t = NOISE * max(0, 1 - t / (STEPS * 0.6))

        grad = latent_score_fn(z, prompt)
        grad = grad / (grad.abs().mean() + 1e-8)
        z = z + LR * grad + torch.randn_like(z) * noise_t

        if t % 50 == 0:
            with torch.no_grad():
                img = decode_latent(z)
                img_224 = torch.nn.functional.interpolate(img, (224,224), mode="bilinear", align_corners=False)
                img_norm = (img_224 - CLIP_MEAN) / CLIP_STD
                i_f = clip_model.encode_image(img_norm)
                t_f = clip_model.encode_text(tok)
                i_f = i_f / i_f.norm(dim=-1, keepdim=True)
                t_f = t_f / t_f.norm(dim=-1, keepdim=True)
                sim = (i_f * t_f).sum().item()
            sims.append((t, sim))
            print("  step " + str(t) + "/" + str(STEPS) + " | CLIP: " + str(round(sim, 4)))

    final_img = decode_latent(z)
    results.append((prompt, final_img.cpu(), sims))
    gain = sims[-1][1] - sims[0][1]
   

In [ ]:
gains  = [r[2][-1][1]-r[2][0][1] for r in results]
finals = [r[2][-1][1] for r in results]

lines = [
   
    "",
    "Results:",
    "Healthy lungs  CLIP: " + str(round(finals[0],3)) + " (+" + str(round(gains[0],4)) + ")",
    "Pneumonia      CLIP: " + str(round(finals[1],3)) + " (+" + str(round(gains[1],4)) + ")",
    "Lung nodule    CLIP: " + str(round(finals[2],3)) + " (+" + str(round(gains[2],4)) + ")",
    "",]
caption = "\n".join(lines)
print(caption)

In [ ]:


BG, ACCENT, DIM = "#0d0d0d", "#00ff88", "#888888"
LABELS = ["Healthy Lungs", "Pneumonia", "Lung Nodule"]

fig = plt.figure(figsize=(20, 14), facecolor=BG)
fig.suptitle(
    "SCORENET-ZERO v2  |  Latent Space  |  ZERO Training  |  VAE + CLIP + Steins Identity",
    color="white", fontsize=14, fontweight="bold", y=0.98
)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.3,
                       top=0.92, bottom=0.08, left=0.05, right=0.97)

for i, (prompt, img_t, sims) in enumerate(results):

    ax_img = fig.add_subplot(gs[0, i])
    ax_img.set_facecolor(BG)
    img_np = img_t.squeeze(0).permute(1,2,0).numpy()
    img_np = np.clip(img_np, 0, 1)
    gray   = 0.299*img_np[:,:,0] + 0.587*img_np[:,:,1] + 0.114*img_np[:,:,2]
    ax_img.imshow(gray, cmap="bone", vmin=0, vmax=1)
    ax_img.set_title("Image " + str(i+1) + ": " + LABELS[i],
                     color=ACCENT, fontsize=13, fontweight="bold", pad=10)
    gain  = sims[-1][1] - sims[0][1]
    line1 = "CLIP: " + str(round(sims[-1][1], 3))
    line2 = "+" + str(round(gain, 4))
    badge = line1 + "\n" + line2
    ax_img.text(0.04, 0.05, badge,
        transform=ax_img.transAxes, color=ACCENT, fontsize=9,
        fontweight="bold", fontfamily="monospace",
        bbox=dict(facecolor="black", edgecolor=ACCENT, boxstyle="round,pad=0.3"))
    ax_img.axis("off")
    for sp in ax_img.spines.values():
        sp.set_edgecolor(ACCENT)
        sp.set_linewidth(2)

    ax_c = fig.add_subplot(gs[1, i])
    ax_c.set_facecolor("#111111")
    for sp in ax_c.spines.values():
        sp.set_edgecolor("#333333")
    xs = [s for s,_ in sims]
    ys = [v for _,v in sims]
    ax_c.fill_between(xs, min(ys), ys, alpha=0.15, color=ACCENT)
    ax_c.plot(xs, ys, color=ACCENT, linewidth=2.5)
    ax_c.scatter(xs[0],  ys[0],  color="white", s=70, zorder=5)
    ax_c.scatter(xs[-1], ys[-1], color=ACCENT,  s=90, zorder=5)
    ax_c.annotate(str(round(ys[0], 3)),
        xy=(xs[0], ys[0]), xytext=(15,-18), textcoords="offset points",
        color="white", fontsize=8,
        arrowprops=dict(arrowstyle="->", color="white", lw=1))
    ax_c.annotate(str(round(ys[-1], 3)),
        xy=(xs[-1], ys[-1]), xytext=(-40,12), textcoords="offset points",
        color=ACCENT, fontsize=8, fontweight="bold",
        arrowprops=dict(arrowstyle="->", color=ACCENT, lw=1))
    ax_c.set_title("Quality Curve - Image " + str(i+1),
                   color="white", fontsize=10, fontweight="bold", pad=8)
    ax_c.set_xlabel("Latent Step", color=DIM, fontsize=9)
    ax_c.set_ylabel("CLIP Similarity", color=DIM, fontsize=9)
    ax_c.tick_params(colors=DIM, labelsize=8)
    ax_c.set_xlim(0, STEPS)
    ax_c.grid(True, alpha=0.1, color="white")

gains  = [r[2][-1][1]-r[2][0][1] for r in results]
footer = ("VAE: stabilityai/sd-vae-ft-mse (frozen)  |  "
          "CLIP: ViT-B/32 (frozen)  |  "
          "Kernel: 100 real X-rays  |  "
          "Params trained: 0  |  "
          "Gains: +" + str(round(gains[0],4)) + " | +" + str(round(gains[1],4)) + " | +" + str(round(gains[2],4)))

fig.text(0.5, 0.01, footer, color=ACCENT, fontsize=8.5, ha="center",
         fontfamily="monospace",
         bbox=dict(boxstyle="round,pad=0.5", facecolor="#111111",
                   edgecolor=ACCENT, linewidth=1.2))

plt.savefig("scorenet_zero_v2.png", dpi=160,
            bbox_inches="tight", facecolor=BG, edgecolor="none")
plt.show()
